In [1]:
# ============================================================
# Grazioso Salvare Rescue Animal Dashboard
# CS 499 Capstone - Software Design and Engineering Enhancement
# ============================================================

# Dash components
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output

# Data processing
import pandas as pd

# Standard library
import base64

# Project modules
from animal_shelter import AnimalShelter
from rescue_filters import get_rescue_query
from visualization import create_breed_chart, create_animal_map
from candidate_scoring import rank_candidates


# ============================================================
# Database Connection
# ============================================================

# AnimalShelter loads the MongoDB connection settings from
# environment variables rather than storing credentials in code.
db = AnimalShelter()


# ============================================================
# Initial Data Load
# ============================================================

try:
    # Load all animal records for the initial dashboard view.
    records = db.read({})

    if records:
        df = pd.DataFrame.from_records(records)

        # Remove MongoDB's ObjectId before sending records to Dash.
        if "_id" in df.columns:
            df.drop(columns=["_id"], inplace=True)

    else:
        print("No animal records were found in the database.")
        df = pd.DataFrame()

except Exception as error:
    print(f"Unable to load animal records: {error}")
    df = pd.DataFrame()

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


# ============================================================
# Dashboard Layout
# ============================================================

# Create the Dash application.
app = Dash(__name__)


# ============================================================
# Dashboard Logo
# ============================================================

# Encode the Grazioso Salvare logo so it can be displayed
# directly in the dashboard.
image_filename = "Grazioso Salvare Logo.png"

encoded_image = base64.b64encode(
    open(image_filename, "rb").read()
).decode()

# ============================================================
# Dashboard Interface
# ============================================================

app.layout = html.Div([

    # Dashboard header
html.Div([
    html.Img(
        src=f"data:image/png;base64,{encoded_image}",
        style={"height": "150px"}
    ),
    html.H1("Grazioso Salvare Rescue Animal Dashboard"),
    html.H4("Jason Guiste - CS 499 Capstone")
], style={"textAlign": "center"}),
    

    html.Hr(),

# Rescue type filtering controls
html.Div([
    html.H3("Rescue Type"),

    dcc.RadioItems(
        id="filter-type",
        options=[
            {
                "label": "Water Rescue",
                "value": "Water Rescue"
            },
            {
                "label": "Mountain or Wilderness Rescue",
                "value": "Mountain or Wilderness Rescue"
            },
            {
                "label": "Disaster or Individual Tracking",
                "value": "Disaster or Individual Tracking"
            },
            {
                "label": "Reset",
                "value": "Reset"
            }
        ],
        value="Reset",
        labelStyle={
            "display": "inline-block",
            "marginRight": "20px"
        }
    )
], style={"textAlign": "center"}),

html.Hr(),

# ============================================================
# Animal Data Table
# ============================================================

dash_table.DataTable(
    id="datatable-id",

    columns=[
    {
        "name": column,
        "id": column,
        "deletable": False,
        "selectable": True
    }
    for column in df.columns
] + [
    {
        "name": "Suitability Score",
        "id": "suitability_score",
        "deletable": False,
        "selectable": True,
        "type": "numeric"
    }
],
    data=df.to_dict("records"),

    # Allow the user to select one animal at a time.
    row_selectable="single",
    selected_rows=[0],

    # Table navigation and filtering.
    page_action="native",
    page_current=0,
    page_size=10,
    sort_action="native",
    sort_mode="multi",
    filter_action="native",

    # Table formatting.
    style_table={
        "overflowX": "auto"
    },
    style_cell={
        "textAlign": "left",
        "minWidth": "100px",
        "width": "150px",
        "maxWidth": "200px"
    },
    style_header={
        "fontWeight": "bold"
    }
),

html.Br(),
html.Hr(),


# ============================================================
# Dashboard Visualizations
# ============================================================

# Display the breed chart and animal location map side by side.
html.Div(
    className="row",
    style={
        "display": "flex",
        "width": "100%"
    },
    children=[

        # Breed distribution chart
        html.Div(
            id="graph-id",
            className="col s12 m6",
            style={
                "width": "50%"
            }
        ),

        # Animal location map
        html.Div(
            id="map-id",
            className="col s12 m6",
            style={
                "width": "50%"
            }
        )
    ]
)

])

# ============================================================
# Dashboard Callbacks
# ============================================================

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """Update and rank animal records for the selected rescue type."""

    try:
        # Retrieve all animal records from MongoDB.
        records = db.read({})

        if not records:
            return []

        # Rank candidates when a rescue category is selected.
        if filter_type != "Reset":
            records = rank_candidates(records, filter_type)

        data = pd.DataFrame.from_records(records)

        # Remove MongoDB's ObjectId before sending data to Dash.
        if "_id" in data.columns:
            data.drop(columns=["_id"], inplace=True)

        return data.to_dict("records")

    except Exception as error:
        print(f"Unable to update dashboard: {error}")
        return []

# ============================================================
# Breed Chart
# ============================================================

@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data")
)
def update_graph(view_data):
    """Update the breed chart when the table data changes."""

    if view_data is None:
        filtered_data = df
    else:
        filtered_data = pd.DataFrame.from_dict(view_data)

    if filtered_data.empty:
        return []

    return [
        dcc.Graph(
            figure=create_breed_chart(filtered_data)
        )
    ]
    
# ============================================================
# Table Styling
# ============================================================

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns")
)
def update_styles(selected_columns):
    """Highlight columns selected by the user."""

    if selected_columns is None:
        selected_columns = []

    return [
        {
            "if": {"column_id": column},
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


# ============================================================
# Animal Location Map
# ============================================================

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(view_data, selected_rows):
    """Update the map to display the selected animal's location."""

    if view_data is None:
        filtered_data = df
    else:
        filtered_data = pd.DataFrame.from_dict(view_data)

    if filtered_data.empty:
        return []

    # Display the first animal when no row has been selected.
    if selected_rows is None or len(selected_rows) == 0:
        row = 0
    else:
        row = selected_rows[0]

    return [
        create_animal_map(filtered_data, row)
    ]

# ============================================================
# Run Application
# ============================================================

app.run(debug=True)
